# Multiscale CNN: train model độc lập

Giữ feature CLIP có sẵn và LGT của VadCLIP; LGT được khởi tạo mới. Không dùng checkpoint đã train, text, nhánh A/C hoặc loss frame. Tám nhánh Conv1D k=2/3/4/5/6/7/8/9, mỗi nhánh 32 kernel, không GELU sau CNN, ghép 256 channels → max theo thời gian → FC 2 logits → softmax để dự đoán video. Activation trong LGT giữ nguyên. Kernel chẵn dùng same padding lệch: trái floor((k-1)/2), phải floor(k/2), giữ nguyên chiều dài response.

Train dùng Cross-Entropy trên logits video cộng L1 trên Δw = weight Abnormal − weight Normal, lambda=1e-4; không phạt bias. Frame score vẫn dùng toàn bộ kernel và cùng FC áp vào response trước pooling, không dùng nhãn frame để train. Train 10 epoch, evaluate 5/7/10; epoch 10 là kết quả chính, không chọn theo test AUC.

LGT giữ phép tính padding gốc. CNN loại feature padding và max pooling chỉ xét vị trí hợp lệ. Khi test, LGT chạy các khối 256; CNN chạy trên chuỗi LGT đã ghép theo thời gian, pooling một lần cho cả video. Các crop/flip không được nối thành chuỗi.


In [ ]:
import os
import sys
import json
import csv
import shlex
import shutil
import subprocess
from pathlib import Path
from google.colab import drive
import torch

# Rời cwd trên Drive trước khi mount lại.
os.chdir('/content')
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP/src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP/list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ARCHIVE = PROJECT_ROOT / 'UCFClipFeatures.zip'
RUN_TAG = 'mscnn_k2_9_delta_l1e4_s234'
CNN_ACTIVATION = 'none'
LAMBDA_FC_L1 = 1e-4
FC_L1_TARGET = 'delta'
KERNEL_SIZES = [2, 3, 4, 5, 6, 7, 8, 9]
CHANNELS_PER_BRANCH = 32
MAX_EPOCH = 10
EVALUATE_EPOCHS = [5, 7, 10]
SEED = 234
# Chỉ dùng baseline tự train, KHÔNG dùng model_ucf.pth của tác giả.
BASELINE_MODEL_PATH = SRC_DIR / 'model/model_ucf_paper_scratch.pth'
RESULT_ROOT = PROJECT_ROOT / 'code/results'
TRAIN_DIR = RESULT_ROOT / 'train/multiscale_cnn' / RUN_TAG
EVAL_DIR = RESULT_ROOT / 'evaluate/multiscale_cnn' / RUN_TAG
ANALYZE_DIR = RESULT_ROOT / 'analyze/multiscale_cnn' / RUN_TAG
CHECKPOINT_DIR = SRC_DIR / 'model/epoch_checkpoints_multiscale_cnn' / RUN_TAG

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime và kiểm tra PyTorch CUDA trước khi chạy.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    if not FEATURE_ARCHIVE.is_file():
        raise FileNotFoundError(FEATURE_ARCHIVE)
    local_zip = Path('/content') / FEATURE_ARCHIVE.name
    if not local_zip.is_file() or local_zip.stat().st_size != FEATURE_ARCHIVE.stat().st_size:
        shutil.copy2(FEATURE_ARCHIVE, local_zip)
    subprocess.run(['unzip', '-oq', str(local_zip), '-d', '/content'], check=True)
required = [FEATURE_ROOT, LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
            LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', LIST_DIR / 'gt_ucf.npy']
required += [SRC_DIR / name for name in (
    'model_multiscale_cnn.py', 'ucf_train_multiscale_cnn.py',
    'ucf_evaluate_multiscale_cnn.py', 'test_multiscale_cnn.py',
    'model.py', 'ucf_cnn_common.py', 'utils/layers.py', 'utils/tools.py')]
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
if any(e < 1 or e > MAX_EPOCH for e in EVALUATE_EPOCHS):
    raise ValueError('EVALUATE_EPOCHS phải nằm trong số epoch training.')

def run_command(command, log_path=None):
    print('Running:', shlex.join([str(p) for p in command]), flush=True)
    env = {**os.environ, 'PYTHONHASHSEED': str(SEED), 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
    log = None
    if log_path is not None:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        log = Path(log_path).open('a', encoding='utf-8')
    try:
        with subprocess.Popen(command, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                if log:
                    log.write(line)
                    log.flush()
            if process.wait():
                raise RuntimeError('Lệnh thất bại; traceback gốc đã được in phía trên.')
    finally:
        if log:
            log.close()

print('GPU:', torch.cuda.get_device_name(0))
print('Kernel sizes:', KERNEL_SIZES, '| Total channels:', len(KERNEL_SIZES) * CHANNELS_PER_BRANCH)
print('Train:', TRAIN_DIR, '\nEvaluate:', EVAL_DIR, '\nAnalyze:', ANALYZE_DIR)


## Kiểm thử trước khi train

Kiểm tra CNN, gradient LGT, tương đương phép tính LGT gốc, padding, video dài và một epoch CPU trên dữ liệu giả. Không dùng dữ liệu test để train.


In [ ]:
run_command([sys.executable, '-m', 'unittest', 'test_multiscale_cnn'])


## Training từ scratch

LR 1e-5, AdamW weight decay 0.01, giảm LR tại epoch 4/8, batch 64 Normal + 64 Abnormal, gradient clipping 1.0, không AMP, deterministic. Loss = CE + lambda × tổng |weight Abnormal − weight Normal|; in CE, L1, lambda × L1 và thống kê |delta weight| mỗi epoch. Gradient CE và L1 trên FC được đo trước clipping ở batch đầu mỗi epoch. Toàn bộ danh sách 16.100 view được dùng làm tập train; batch pairing/drop-last theo baseline. Lưu mọi epoch, không early stopping hoặc chọn best test AUC.

Nếu chạy lại cell, chỉ bỏ qua train khi đã hoàn tất và cấu hình khớp. Muốn chạy lại từ đầu, đổi RUN_TAG. Run dở dang không tự resume hoặc ghi đè.


In [ ]:
options = {
    'feature_root': str(FEATURE_ROOT),
    'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
    'test_list': str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    'gt_path': str(LIST_DIR / 'gt_ucf.npy'),
    'results_dir': str(TRAIN_DIR), 'evaluate_dir': str(EVAL_DIR),
    'analyze_dir': str(ANALYZE_DIR), 'epoch_checkpoint_dir': str(CHECKPOINT_DIR),
    'kernel_sizes': KERNEL_SIZES, 'channels_per_branch': CHANNELS_PER_BRANCH,
    'cnn_activation': CNN_ACTIVATION,
    'lambda_fc_l1': LAMBDA_FC_L1,
    'fc_l1_target': FC_L1_TARGET,
    'max_epoch': MAX_EPOCH, 'evaluate_epochs': EVALUATE_EPOCHS,
    'seed': SEED, 'batch_size': 64, 'lr': 1e-5, 'weight_decay': 0.01,
    'scheduler_milestones': [4, 8], 'scheduler_rate': 0.1,
    'grad_clip_norm': 1.0, 'num_workers': 2, 'visual_length': 256,
    'expected_train_views': 16100, 'expected_test_videos': 290, 'micro_batch': 8,
    'device': 'cuda',
}
config_path = TRAIN_DIR / 'config.json'
complete_path = TRAIN_DIR / 'completed.json'
final_path = CHECKPOINT_DIR / f'epoch_{MAX_EPOCH:03d}.pth'
if config_path.is_file():
    previous = json.loads(config_path.read_text())
    changed = [key for key, value in options.items() if previous.get(key) != value]
    if changed:
        raise RuntimeError('Cấu hình khác run đã lưu. Đổi RUN_TAG. Khác: ' + ', '.join(changed))
    if not complete_path.is_file() or not final_path.is_file():
        raise RuntimeError('Run chưa hoàn tất. Đổi RUN_TAG để train lại từ scratch; không tự ghi đè.')
    print('Run hoàn tất đúng cấu hình; bỏ qua train:', final_path)
else:
    command = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_multiscale_cnn.py')]
    for key, value in options.items():
        command.append('--' + key.replace('_', '-'))
        command.extend(str(v) for v in (value if isinstance(value, list) else [value]))
    run_command(command, TRAIN_DIR / 'console.log')


## Đọc kết quả và so sánh baseline tự train

Cell này có thể chạy riêng sau training để evaluate lại checkpoint đã có, không train lại. Kết quả gồm frame AUC/AP, Ano-AUC, video AUC/AP/accuracy và diagnostics. Không có AUC2/mAP vì không có nhánh alignment phân loại hành động. Baseline được đánh giá bằng cùng test list và cách chuyển feature score về frame; không suy ra video score cho baseline.


In [ ]:
def eval_command(checkpoint, ev, an, baseline=False):
    return [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_multiscale_cnn.py'),
            '--baseline-model-path' if baseline else '--model-path', str(checkpoint),
            '--feature-root', str(FEATURE_ROOT),
            '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
            '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
            '--evaluate-dir', str(ev), '--analyze-dir', str(an), '--expected-videos', '290']

summary = []
if BASELINE_MODEL_PATH is not None and Path(BASELINE_MODEL_PATH).is_file():
    baseline_dir = EVAL_DIR / 'baseline'
    run_command(eval_command(BASELINE_MODEL_PATH, baseline_dir, ANALYZE_DIR / 'baseline', True))
    summary.append(json.loads((baseline_dir / 'metrics.json').read_text()))
else:
    print('Chưa có baseline tự train tại path đã chọn; bỏ qua so sánh, không dùng checkpoint tác giả.')
for epoch in EVALUATE_EPOCHS:
    tag = f'epoch_{epoch:03d}'
    path = CHECKPOINT_DIR / f'{tag}.pth'
    if not path.is_file():
        raise FileNotFoundError(path)
    ev, an = EVAL_DIR / tag, ANALYZE_DIR / tag
    if not (ev / 'metrics.json').is_file() or not (an / 'summary.json').is_file():
        run_command(eval_command(path, ev, an))
    summary.append(json.loads((ev / 'metrics.json').read_text()))

columns = ['model_type', 'epoch', 'classifier_auc', 'classifier_ap', 'ano_auc',
           'video_auc', 'video_ap', 'video_accuracy']
EVAL_DIR.mkdir(parents=True, exist_ok=True)
with (EVAL_DIR / 'summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(summary)
print(' | '.join(columns))
for row in summary:
    print(' | '.join(f'{row[c]:.2f}' if isinstance(row.get(c), float) else str(row.get(c, '-')) for c in columns))
print('Epoch chính:', MAX_EPOCH, '| Bảng:', EVAL_DIR / 'summary.csv')
print('Diagnostics: summary.json, kernel_evidence.csv, frame_scores.csv trong', ANALYZE_DIR)


## Phân tích L1 và lựa chọn kernel

Chỉ chạy inference từ hai checkpoint epoch 10, không train lại. So sánh phân bố FC và đóng góp của kernel trên video; cả hai model đều được thử toàn bộ kernel, Top-128/64/32 theo |Δw| và nhóm Δw > 0. Bias FC không đổi; score frame dùng cùng công thức FC trên response trước pooling. Kết quả trên test chỉ mang tính thăm dò, không dùng test để chọn K làm kết quả cuối. Không đối chiếu kernel cùng index giữa hai model.

Chạy cell thiết lập đầu notebook trước, sau đó có thể chạy trực tiếp cell này; không cần chạy cell training.


In [ ]:
from datetime import datetime

# Có thể thay checkpoint cùng kiến trúc và cùng epoch cho các lần phân tích sau.
NO_L1_CHECKPOINT = SRC_DIR / 'model/epoch_checkpoints_multiscale_cnn/mscnn_k2_9_nogelu_s234/epoch_010.pth'
L1_CHECKPOINT = SRC_DIR / 'model/epoch_checkpoints_multiscale_cnn/mscnn_k2_9_delta_l1e4_s234/epoch_010.pth'
SELECTION_TOP_K = [128, 64, 32]
SELECTION_OUTPUT = (RESULT_ROOT / 'analyze/multiscale_kernel_selection' /
                    datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
analyzer = SRC_DIR / 'ucf_analyze_multiscale_selection.py'
for path in (NO_L1_CHECKPOINT, L1_CHECKPOINT, analyzer, FEATURE_ROOT,
             LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', LIST_DIR / 'gt_ucf.npy'):
    if not path.exists():
        raise FileNotFoundError(path)
command = [sys.executable, '-u', str(analyzer),
           '--no-l1-checkpoint', str(NO_L1_CHECKPOINT), '--l1-checkpoint', str(L1_CHECKPOINT),
           '--feature-root', str(FEATURE_ROOT), '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
           '--gt-path', str(LIST_DIR / 'gt_ucf.npy'), '--output-dir', str(SELECTION_OUTPUT),
           '--top-k', *map(str, SELECTION_TOP_K), '--expected-videos', '290',
           '--expected-epoch', '10', '--micro-batch', '8']
run_command(command)
print('Kết quả phân tích:', SELECTION_OUTPUT)
